# 🚀 Huấn Luyện YOLOv8 Object Detection Trên Google Colab (GPU T4 Miễn Phí)

Notebook này giúp bạn huấn luyện mô hình phát hiện đa vật thể (các khối Cube màu) bằng **GPU tốc độ cao trên Google Colab** (chỉ mất ~2-3 phút thay vì hàng giờ trên CPU máy cá nhân).

---
### 📌 Lưu ý trước khi bắt đầu:
1. Vào menu **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Tại mục **Hardware accelerator** (Trình tăng tốc phần cứng), chọn **T4 GPU** rồi bấm **Save**.

### Bước 1: Kiểm tra card đồ họa GPU

In [ ]:
!nvidia-smi

### Bước 2: Cài đặt thư viện Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics

### Bước 3: Tải tập dữ liệu lên Colab
1. Ở cột menu bên trái Colab, bấm vào biểu tượng **📁 Thư mục (Files)**.
2. Kéo thả file `yolo_dataset.zip` từ máy tính của bạn vào đó.
3. Chạy ô code bên dưới để giải nén:

In [ ]:
import os

if os.path.exists('/content/yolo_dataset.zip'):
    !unzip -q /content/yolo_dataset.zip -d /content/yolo_dataset
    print("[+] Đã giải nén yolo_dataset thành công!")
else:
    print("[!] Chưa tìm thấy file yolo_dataset.zip. Hãy kéo thả file vào cột Files bên trái rồi chạy lại ô này!")

### Bước 4: Cập nhật đường dẫn data.yaml cho Colab

In [ ]:
colab_yaml = """path: /content/yolo_dataset
train: images/train
val: images/val

names:
  0: cube_blue
  1: cube_green
  2: cube_red
  3: cube_yellow
"""
with open('/content/yolo_dataset/data.yaml', 'w', encoding='utf-8') as f:
    f.write(colab_yaml)

print("[+] Cấu hình data.yaml hoàn tất:")
print(open('/content/yolo_dataset/data.yaml').read())

### Bước 5: Bắt đầu Huấn luyện với GPU (Chỉ mất khoảng 2 - 3 phút)

In [ ]:
from ultralytics import YOLO

# Tải mô hình nền siêu nhẹ YOLOv8n
model = YOLO('yolov8n.pt')

# Huấn luyện 50 epoch với GPU
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,  # 0 là GPU T4 của Colab
    name='custom_cubes',
    plots=True
)
print("[+] Huấn luyện xong!")

### Bước 6: Xem biểu đồ kết quả đánh giá (Loss & Độ chính xác mAP)

In [ ]:
import matplotlib.pyplot as plt
import cv2

res_plot = "/content/runs/detect/custom_cubes/results.png"
if os.path.exists(res_plot):
    plt.figure(figsize=(15, 8))
    plt.imshow(cv2.cvtColor(cv2.imread(res_plot), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Biểu đồ huấn luyện YOLOv8")
    plt.show()

val_pred = "/content/runs/detect/custom_cubes/val_batch0_pred.jpg"
if os.path.exists(val_pred):
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(cv2.imread(val_pred), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Ảnh dự đoán Bounding Box thực tế")
    plt.show()

### Bước 7: Tải file mô hình `best.pt` về máy tính của bạn
Chạy ô này, trình duyệt sẽ tự động tải file `best.pt` về máy. Sau đó bạn chỉ việc dán nó vào dự án để chạy camera!

In [ ]:
from google.colab import files

best_pt = "/content/runs/detect/custom_cubes/weights/best.pt"
if os.path.exists(best_pt):
    print("[*] Đang tải best.pt về máy...")
    files.download(best_pt)
else:
    print("[!] Không tìm thấy best.pt, vui lòng kiểm tra lại thư mục runs.")